In [0]:
# Date-range presets for data loading
DATE_RANGES = {
    # "full": {"start": "2022-06-30", "end": "2024-07-01", "mob_months": 25},
     "full": {"start": "2022-06-30", "end": "2026-07-01", "mob_months": 13},
    "recent": {"start": "2025-01-31", "end": "2026-07-01", "mob_months": 37},
}

# Bad-definition candidates (min_bucket follows the DELINQUENCYBUCKET encoding:
#   0=current, 1=1-29 DPD, 2=30-59 DPD, 3=60-89 DPD, 4=90+ DPD)
BAD_DEFINITIONS = {
    "Ever60DPD_12M": {"min_bucket": 3, "window_months": 12},
    "Ever30DPD_6M":  {"min_bucket": 2, "window_months": 6},
    "Ever90DPD_12M": {"min_bucket": 4, "window_months": 12},
}

# Target generation: 12-month performance window
TARGET_WINDOW_MONTHS = 12

# Minimum OPEN snapshots a card must have to be retained
# MIN_OPEN_SNAPSHOTS = 3

# Columns needed downstream (avoids OOM when converting full 88-col frame to pandas)
ANALYSIS_COLS = [
    "SNAPSHOT_DATE", "CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE", "APPRECDDATE",
    "DELINQUENCYBUCKET", "CURRENT_STATUS", "REFERENCE_NUMBER", "FTS_Ref_No",
    "EMID_REFERENCENUMBER", "GENERALSTAT",
    # Filter / exclusion columns (kept for downstream analysis)
    "CARD_BUSINESS_SEGMENT", "CUSTOMER_SEGMENT", "VIP", "RULING_FAMILY_FLAG",
    "PRIMARYCARD", "F_SEGMENT_NEW", "SEGMENT_POLICY", "CLASSCODE", "ACTIVE_REPORTING",
]

ID_COLS = [
    "CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE", "APPRECDDATE",
    "REFERENCE_NUMBER", "FTS_Ref_No", "EMID_REFERENCENUMBER", "GENERALSTAT",
    # Filter / exclusion columns (carried through the pipeline)
    "CARD_BUSINESS_SEGMENT", "CUSTOMER_SEGMENT", "VIP", "RULING_FAMILY_FLAG",
    "PRIMARYCARD", "F_SEGMENT_NEW", "SEGMENT_POLICY", "CLASSCODE", "ACTIVE_REPORTING",
]

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col

In [0]:
def load_and_clean_credit_card_data(date_start, date_end, mob_months=37):
    """
    Load credit card history and apply business exclusion filters.

    Parameters
    ----------
    date_start : str   – APPROVEDATE lower bound (exclusive), e.g. '2022-06-30'
    date_end   : str   – APPROVEDATE upper bound (exclusive), e.g. '2025-07-01'
    mob_months : int   – months-on-book observation window (default 37)

    Returns
    -------
    pyspark.sql.DataFrame  – cleaned Spark DataFrame with exclusions applied.
    """
    # Step 1: Load raw data within the observation window
    df = spark.sql(f"""
        SELECT *
        FROM mis_automation.silver_credit_card.vw_credit_card_history
        WHERE SNAPSHOT_DATE >= APPROVEDATE
          AND SNAPSHOT_DATE < add_months(APPROVEDATE, {mob_months})
          AND APPROVEDATE > '{date_start}'
          AND APPROVEDATE < '{date_end}'
          AND UPPER(TRIM(CARD_BUSINESS_SEGMENT)) IN ('CARDS')
          AND UPPER(TRIM(CUSTOMER_SEGMENT)) = 'SALARIED'
    """)

    # Step 2: Exclusion conditions
    exclude_cond = (
        (F.col("VIP") == "1")                                                             # 3a: VIP customers
        | (F.upper(F.trim(F.col("RULING_FAMILY_FLAG"))) == "Y")                             # 3b: Ruling family
        | (F.col("PRIMARYCARD") == 0)                                                      # 7: Supplementary cards
        | (F.upper(F.trim(F.col("F_SEGMENT_NEW"))) == "SALARIED-IM")                        # 10a: Salaried-IM
        | (F.upper(F.trim(F.col("SEGMENT_POLICY"))).isin(
              "PL CUSTOMERS", "AUTO-LOAN CUSTOMERS", "MORTGAGE CXMS", "STAFF"))          # 10b: Unwanted segment policies
        |(F.upper(F.trim(F.col("GENERALSTAT"))).isin('FRUD')))
     
    inclusive_cond = (
        (F.col("CLASSCODE").isNull()| ~F.trim(F.col("CLASSCODE")).isin("STAFF", "SME"))   # 1, 8: SME/STAFF class code
        & (F.upper(F.trim(F.col("CARD_BUSINESS_SEGMENT"))).isin("CARDS"))                  # 2: CARDS segment
        & (F.upper(F.trim(F.col("CUSTOMER_SEGMENT"))) == "SALARIED")                       # 6: Salaried
        & (F.upper(F.trim(F.col("ACTIVE_REPORTING"))) == "Y")                              # 4: Active reporting
    )        

    # df_clean = df.filter(~exclude_cond).filter(inclusive_cond)  # disabled
    df_clean = df  # Exclusion/inclusion filters skipped

    return df_clean

def generate_target(df, window_months=TARGET_WINDOW_MONTHS):
    """
    Assign one target label per CREDITCARDNO based on the max
    DELINQUENCYBUCKET within the first `window_months` after approval.

    Target mapping
    --------------
    -2 : immature (fewer than `window_months` unique snapshot months)
     0 : good    (max bucket == 0)
    -1 : indeterminate (max bucket in 1-3)
     1 : bad     (max bucket >= 4, i.e. 90+ DPD)
    """
    df = df.copy()
    df["SNAPSHOT_DATE"] = pd.to_datetime(df["SNAPSHOT_DATE"])
    df["APPROVEDATE"]   = pd.to_datetime(df["APPROVEDATE"])

    mask = (
        (df["SNAPSHOT_DATE"] >= df["APPROVEDATE"])
        & (df["SNAPSHOT_DATE"] < df["APPROVEDATE"] + pd.DateOffset(months=window_months))
    )
    window_df = df.loc[mask].copy()
    window_df["_snap_month"] = window_df["SNAPSHOT_DATE"].dt.to_period("M")

    agg = (
        window_df
        .groupby("CREDITCARDNO")
        .agg(max_bucket=("DELINQUENCYBUCKET", "max"),
             unique_months=("_snap_month", "nunique"))
        .reset_index()
    )

    conditions = [
        agg["unique_months"] < window_months,  # -2: immature
        agg["max_bucket"] >= 4,                 #  1: bad 3 is used
        agg["max_bucket"] == 0,                 #  0: good
        agg["max_bucket"].between(1, 3),        # -1: indeterminate
    ]
    agg["target"] = np.select(conditions, [-2, 1, 0, -1], default=np.nan)

    if agg["target"].isna().any():
        n_drop = agg["target"].isna().sum()
        print(f"WARNING: {n_drop} rows matched no target condition — dropped.")
        agg = agg.dropna(subset=["target"])
    agg["target"] = agg["target"].astype(int)

    id_df = df[ID_COLS].drop_duplicates(subset="CREDITCARDNO")
    result = id_df.merge(agg[["CREDITCARDNO", "target"]], on="CREDITCARDNO", how="inner")
    result["REFERENCE_NUMBER"] = (
        result["REFERENCE_NUMBER"].astype(str).str.replace(r'\.0$', '', regex=True)
    )
    return result

def load_aecb_feature_store(df_target):
    """
    Load the AECB feature-store table, filter to reference numbers
    present in `df_target`, merge, and split into thin-file subset.

    Optimised: uses a broadcast Spark join instead of .isin() with a
    Python list, which pushes the filter into the scan and avoids
    serialising thousands of reference numbers to every executor.

    Returns
    -------
    aecb_merged : pd.DataFrame – AECB features joined with target
    df_thin     : pd.DataFrame – subset where file_type == 'Thin File'
    """
    # ── 1. Build a small Spark DF of unique reference numbers ────────────
    ref_pd = (
        df_target[["REFERENCE_NUMBER"]]
        .drop_duplicates()
        .rename(columns={"REFERENCE_NUMBER": "ref_num"})
    )
    ref_pd["ref_num"] = ref_pd["ref_num"].astype(str)
    ref_spark = spark.createDataFrame(ref_pd)
    ref_spark.createOrReplaceTempView("_aecb_ref_nums")

    # ── 2. Push filter into Spark via broadcast join ────────────────────
    aecb_filtered = spark.sql("""
        SELECT /*+ BROADCAST(r) */ a.*
        FROM mis_automation.silver_bureauone.slv_credit_card_aecb_feature_store a
        INNER JOIN _aecb_ref_nums r
               ON CAST(a.Reference_Number AS STRING) = r.ref_num
    """)
    aecb_pd = aecb_filtered.toPandas()

    # ── 2b. Prefix AECB feature columns with 'aecb_' ───────────────────
    skip_cols = {"Reference_Number"}  # columns used for joining — don't rename
    aecb_pd = aecb_pd.rename(
        columns={c: f"aecb_{c}" for c in aecb_pd.columns if c not in skip_cols}
    )

    # ── 3. Pandas merge with target columns ─────────────────────────────
    aecb_pd["Reference_Number"] = aecb_pd["Reference_Number"].astype(str)
    df_target["REFERENCE_NUMBER"] = df_target["REFERENCE_NUMBER"].astype(str)

    aecb_merged = aecb_pd.merge(
        df_target[ID_COLS + ["target"]].drop_duplicates(),
        left_on="Reference_Number",
        right_on="REFERENCE_NUMBER",
        how="inner",
    )
    # ── 3. Use the new file type logic from policy ────────────────────
    df_thin = aecb_merged.loc[aecb_merged["aecb_file_type_revised"] == "Thin File"].reset_index(drop=True)

    print(f"AECB shape   : {aecb_pd.shape}")
    print(f"Merged shape : {aecb_merged.shape}")
    print(f"Thin-file    : {df_thin.shape[0]:,} rows  |  {df_thin['CREDITCARDNO'].nunique():,} cards")
    print(f"\nTarget distribution:\n{aecb_merged['target'].value_counts(dropna=False).sort_index()}")
    return aecb_merged, df_thin

def load_fts_features(df_thin_score, journey="digital"):
    """
    Load FTS features and merge them onto the thin-file scored DataFrame.

    Parameters
    ----------
    df_thin_score : pd.DataFrame
        Thin-file DataFrame with an 'FTS_Ref_No' column to join on.
    journey : str
        Which journey to load. Currently supports 'digital'.
        ('physical' to be added later.)

    Returns
    -------
    pd.DataFrame – df_thin_score enriched with FTS feature columns.
    """
    # ── 1. Select source table based on journey ──────────────────────────
    if journey == "digital":
        table = "mis_automation.silver_ftsconnect.fts_features"
    # elif journey == "physical":
    #     table = "<physical_journey_table>"   # TODO: add physical journey table
    else:
        raise ValueError(f"Unknown journey '{journey}'. Supported: 'digital'.")

    fts_features_sdf = spark.sql(f"SELECT * FROM {table}")
    fts_cols_all = fts_features_sdf.columns  # schema read once

    print(f"Journey            : {journey}")
    print(f"df_thin_score rows : {len(df_thin_score):,}")
    print(f"fts_features rows  : {fts_features_sdf.count():,}")
    print(f"fts_features cols  : {len(fts_cols_all)}")

    # ── 2. Align the join key on both sides ──────────────────────────────
    # Both sources store the key as a 15/16-char string (e.g. '2023090612352531',
    # 'SRQ2505230004183'). 15-char keys are valid on both sides – do NOT pad.
    # Normalise: cast to string, trim, upper-case, strip a pandas '.0' suffix,
    # and treat ''/'NONE'/'NAN'/'NULL' as null.
    def _norm_key(c):
        k = F.regexp_replace(F.upper(F.trim(c.cast("string"))), r"\.0$", "")
        return F.when(k.isin("", "NONE", "NAN", "NULL"), F.lit(None)).otherwise(k)

    # Pandas side: force key columns to str/None so Spark infers StringType
    pdf = df_thin_score.copy()
    for c in ["FTS_Ref_No", "EMID_REFERENCENUMBER"]:
        if c in pdf.columns:
            pdf[c] = pdf[c].map(lambda v: None if pd.isna(v) else str(v))
    pdf["_row_id"] = np.arange(len(pdf))

    left_sdf = (
        spark.createDataFrame(pdf)
        .withColumn("_fts_key", _norm_key(F.col("FTS_Ref_No")))
        .withColumn("_emid_key", _norm_key(F.col("EMID_REFERENCENUMBER")))
    )
    right_sdf = (
        fts_features_sdf
        .withColumn("_fts_key_r", _norm_key(F.col("fts_ref_no")))
        .withColumn("_fts_emid_r", _norm_key(F.col("fts_master_id_number")))
        .drop("fts_ref_no")   # avoid case-insensitive clash with FTS_Ref_No
        .filter(F.col("_fts_key_r").isNotNull())
    )

    # ── 3. Left-join on the normalised key ───────────────────────────────
    joined_sdf = left_sdf.join(
        right_sdf, left_sdf["_fts_key"] == right_sdf["_fts_key_r"], how="left"
    )

    # ── 4. Resolve duplicate fts_ref_no (19 keys map to 2 different EMIDs) ─
    # Keep one FTS row per input row: prefer the row whose
    # fts_master_id_number matches the applicant EMID, then the latest statement.
    w = Window.partitionBy("_row_id").orderBy(
        (F.col("_fts_emid_r") == F.col("_emid_key")).desc_nulls_last(),
        F.col("fts_statement_create_date").desc_nulls_last(),
    )
    joined_sdf = (
        joined_sdf
        .withColumn("_n_cand", F.count("_fts_key_r").over(Window.partitionBy("_row_id")))
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
    )

    # ── 5. Collect back to pandas and report ─────────────────────────────
    existing_cols = set(df_thin_score.columns)
    df_merged = joined_sdf.orderBy("_row_id").toPandas()

    n_has_key  = int(df_merged["_fts_key"].notna().sum())
    n_matched  = int(df_merged["_fts_key_r"].notna().sum())
    n_multi    = int((df_merged["_n_cand"] > 1).sum())
    n_emid_mis = int((df_merged["_fts_key_r"].notna()
                      & df_merged["_emid_key"].notna()
                      & (df_merged["_fts_emid_r"] != df_merged["_emid_key"])).sum())

    df_merged = df_merged.drop(columns=["_row_id", "_fts_key", "_emid_key", "_fts_key_r",
                                        "_fts_emid_r", "_n_cand", "_rn"])
    assert len(df_merged) == len(df_thin_score), "Row fan-out after FTS join"
    new_cols = [c for c in df_merged.columns if c not in existing_cols]

    print(f"\nMerged shape            : {df_merged.shape}")
    print(f"Rows with valid FTS key : {n_has_key:,}")
    print(f"Rows matched to FTS     : {n_matched:,} ({n_matched / max(len(df_merged), 1):.2%} of all, "
          f"{n_matched / max(n_has_key, 1):.2%} of keyed)")
    print(f"Rows with >1 FTS row    : {n_multi:,} (resolved: EMID match, then latest statement)")
    print(f"Matched but EMID differs: {n_emid_mis:,}")
    if new_cols:
        print(f"\nNew columns from fts_features ({len(new_cols)}):")
        for i, c in enumerate(new_cols, 1):
            print(f"  {i:3d}. {c}")
    else:
        print("No new columns added.")

    return df_merged

def load_fts_features_physical(df_thin_score):
    """
    EMID fallback: for rows without FTS_Ref_No, match FTS features
    via EMID_REFERENCENUMBER = fts_master_id_number.

    Keeps the earliest fts_statement_create_date per CREDITCARDNO
    within a [-7d, +30d) window around APPRECDDATE.

    Parameters
    ----------
    df_thin_score : pd.DataFrame
        Thin-file DataFrame (already enriched by the digital journey).
        Must contain 'FTS_Ref_No', 'EMID_REFERENCENUMBER', 'APPRECDDATE',
        and 'CREDITCARDNO'.

    Returns
    -------
    pd.DataFrame – df_thin_score with FTS columns back-filled for rows
                   that had no FTS_Ref_No but matched on EMID.
    """
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    # ── 0. Guard: ensure CREDITCARDNO is a column ────────────────────────
    if "CREDITCARDNO" not in df_thin_score.columns and df_thin_score.index.name == "CREDITCARDNO":
        df_thin_score = df_thin_score.reset_index()

    # Clean up leftover __emid columns from a previous partial run
    emid_leftover = [c for c in df_thin_score.columns if c.endswith("__emid")]
    if emid_leftover:
        df_thin_score.drop(columns=emid_leftover, inplace=True)
        print(f"Cleaned up {len(emid_leftover)} leftover __emid columns")

    # ── 1. Identify rows where FTS_Ref_No is not available ───────────────
    no_fts_mask = (
        df_thin_score["FTS_Ref_No"].isna()
        | df_thin_score["FTS_Ref_No"].astype(str).isin(["None", "", "nan"])
    )
    df_no_fts = (
        df_thin_score
        .loc[no_fts_mask, ["CREDITCARDNO", "EMID_REFERENCENUMBER", "APPRECDDATE","APPROVEDATE"]]
        .drop_duplicates(subset="CREDITCARDNO")
    )
    df_no_fts = df_no_fts[df_no_fts["EMID_REFERENCENUMBER"].notna()].copy()
    df_no_fts["EMID_REFERENCENUMBER"] = df_no_fts["EMID_REFERENCENUMBER"].astype(str)
    print(f"Cases without FTS_Ref_No (with valid EMID): {len(df_no_fts):,}")

    if df_no_fts.empty:
        print("Nothing to backfill — returning df_thin_score unchanged.")
        return df_thin_score

    # ── 1b. Overall EMID match-rate diagnostic ───────────────────────────
    all_emids = (
        df_thin_score
        .loc[df_thin_score["EMID_REFERENCENUMBER"].notna(),
             ["CREDITCARDNO", "EMID_REFERENCENUMBER"]]
        .drop_duplicates(subset="CREDITCARDNO")
    )
    all_emids["EMID_REFERENCENUMBER"] = all_emids["EMID_REFERENCENUMBER"].astype(str)
    all_emids_sdf = spark.createDataFrame(all_emids)

    fts_sdf_check = spark.sql(
        "SELECT DISTINCT fts_master_id_number FROM mis_automation.silver_ftsconnect.fts_features"
    )
    matched_emids = all_emids_sdf.join(
        fts_sdf_check,
        all_emids_sdf["EMID_REFERENCENUMBER"] == fts_sdf_check["fts_master_id_number"],
        how="inner",
    )
    matched_count = matched_emids.count()
    total_emid_count = all_emids_sdf.count()
    print(f"\n── EMID match check (all rows with valid EMID) ──")
    print(f"Total unique CREDITCARDNOs with valid EMID: {total_emid_count:,}")
    print(f"Found in fts_features.fts_master_id_number: {matched_count:,}")
    if total_emid_count:
        print(f"Match rate:                                 {matched_count / total_emid_count:.2%}")
    print(f"Not found:                                  {total_emid_count - matched_count:,}\n")

    # ── 2. Spark lookup DF ───────────────────────────────────────────────
    lookup_sdf = spark.createDataFrame(df_no_fts)

    # ── 3. Join fts_features on EMID ────────────────────────────────────
    fts_sdf = spark.sql("SELECT * FROM mis_automation.silver_ftsconnect.fts_features")
    joined = lookup_sdf.join(
        fts_sdf,
        lookup_sdf["EMID_REFERENCENUMBER"] == fts_sdf["fts_master_id_number"],
        how="inner",
    )

    cards_before_date_filter = joined.select("CREDITCARDNO").distinct().count()
    print(f"── Drop-off Diagnosis ──")
    print(f"Step 3 - Cards matched on EMID (before date filter): {cards_before_date_filter:,}")
    print(f"         Cards NOT found in fts_features at all:     {len(df_no_fts) - cards_before_date_filter:,}")

    # ── 4. Date filter: [-7d, +30d) around APPRECDDATE ──────────────────
    joined_filtered = joined.filter(
        (F.col("fts_statement_create_date") >= F.expr("APPRECDDATE - INTERVAL 60 DAYS"))
        & (F.col("fts_statement_create_date") < F.expr("APPRECDDATE + INTERVAL 30 DAYS"))
        & (F.col("fts_statement_create_date") < F.col("APPROVEDATE"))
    )
    cards_after_date_filter = joined_filtered.select("CREDITCARDNO").distinct().count()
    print(f"Step 4 - Cards surviving [-7d, +30d) date filter:    {cards_after_date_filter:,}")
    print(f"         Cards LOST by date filter:                  {cards_before_date_filter - cards_after_date_filter:,}\n")

    # ── 5. Keep latest fts_statement_create_date per card ───────────────
    w = Window.partitionBy("CREDITCARDNO").orderBy(F.col("fts_statement_create_date").desc())
    fts_emid = (
        joined_filtered
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn", "EMID_REFERENCENUMBER", "APPRECDDATE")
    )
    fts_emid_pd = fts_emid.toPandas()
    print(f"Matched via EMID: {fts_emid_pd['CREDITCARDNO'].nunique():,} cards")

    # ── 6. Back-fill FTS columns into df_thin_score ─────────────────────
    fts_cols = [
        c for c in fts_emid_pd.columns
        if c.startswith("fts_")
        and c != "fts_master_id_number"
        and c in df_thin_score.columns
    ]
    print(f"FTS feature columns to backfill: {len(fts_cols)}")

    fts_lookup = (
        fts_emid_pd.drop_duplicates(subset="CREDITCARDNO")
        .set_index("CREDITCARDNO")[fts_cols]
    )
    cards_to_fill = set(fts_lookup.index)
    fill_mask = no_fts_mask & df_thin_score["CREDITCARDNO"].isin(cards_to_fill)
    print(f"Rows to backfill: {fill_mask.sum():,}")

    for col_name in fts_cols:
        mapping = fts_lookup[col_name].to_dict()
        df_thin_score.loc[fill_mask, col_name] = (
            df_thin_score.loc[fill_mask, "CREDITCARDNO"].map(mapping)
        )

    # ── Summary ──────────────────────────────────────────────────────────
    first_fts = fts_cols[0] if fts_cols else None
    if first_fts:
        after = df_thin_score[first_fts].notna().sum()
        before = after - fill_mask.sum()
        n = len(df_thin_score)
        print(f"\nFTS coverage before EMID backfill: {before:,} / {n:,} ({before / n:.1%})")
        print(f"FTS coverage after  EMID backfill: {after:,} / {n:,} ({after / n:.1%})")
        print(f"Rows gained: {fill_mask.sum():,}")

    return df_thin_score

In [0]:
# ── Run the full pipeline ──────────────────────────────────────────────
cfg = DATE_RANGES["full"]

# 1. Load credit card data (exclusion/inclusion filters disabled)
df_spark = load_and_clean_credit_card_data(cfg["start"], cfg["end"], cfg["mob_months"])
print(f"Raw rows: {df_spark.count():,}")

# 2. Generate target labels
df_pd = df_spark.select(*ANALYSIS_COLS).toPandas()
df_target = generate_target(df_pd)
print(f"\nTarget shape: {df_target.shape}")
print(df_target["target"].value_counts().sort_index())

# 3. Load AECB feature store
aecb_merged, df_thin = load_aecb_feature_store(df_target)

# 4. FTS features – digital journey
df_thin = load_fts_features(df_thin, journey="digital")

# 5. FTS features – physical (EMID backfill)
df_thin = load_fts_features_physical(df_thin)

# 6. Drop duplicate columns (case-insensitive clash) and save
dupes = df_thin.columns[df_thin.columns.str.lower().duplicated(keep="first")]
if len(dupes):
    print(f"Dropping duplicate columns: {list(dupes)}")
    df_thin = df_thin.drop(columns=dupes)

print(f"\nFinal shape: {df_thin.shape}")
sdf = spark.createDataFrame(df_thin)

# 7. Drop NullType (void) columns — all-null cols that Parquet can't store/read
from pyspark.sql.types import NullType
void_cols = [f.name for f in sdf.schema.fields if isinstance(f.dataType, NullType)]
if void_cols:
    print(f"Dropping NullType (void) columns: {void_cols}")
    sdf = sdf.drop(*void_cols)

sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("mis_automation.ds_exploration.acard_cc_data")
print("Saved to mis_automation.ds_exploration.acard_cc_data")

In [0]:
df = spark.sql("SELECT * FROM mis_automation.ds_exploration.acard_cc_data LIMIT 5").toPandas()
df

In [0]:
[x for x in df.columns.tolist() if 'stdde' in x]

In [0]:
[c for c in df.columns.tolist() if 'variance' in c.lower()]

In [0]:
# Columns that are neither fts_ nor aecb_ prefixed
filtered_cols = [x for x in df.columns.tolist() if "fts_" not in x and "aecb_" not in x]

# Explicit ID / status columns to keep
extra_cols = [
    "CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE", "APPRECDDATE",
    "DELINQUENCYBUCKET", "CURRENT_STATUS", "REFERENCE_NUMBER",
    "FTS_Ref_No", "EMID_REFERENCENUMBER", "GENERALSTAT",
    "aecb_file_type_revised", "aecb_file_type", "aecb_file_type_new",
]

# Combine, deduplicate, and sort
final_cols = sorted(set(filtered_cols + extra_cols))
final_cols

In [0]:
spark.sql("SELECT Customer_number FROM mis_automation.silver_fin_db.customer_information LIMIT 5").display()

In [0]:
demo = spark.sql("SELECT * FROM mis_automation.silver_fin_db.customer_information LIMIT 5").toPandas()
alist = demo.columns.tolist()
print(alist.sort())#mis_automation.ds_exploration.acard_cc_data

In [0]:
# Join acard_cc_data (excluding demographic cols) with customer_information on CIF = Customer_number
from pyspark.sql.functions import col, floor, months_between

df_base = spark.table("mis_automation.ds_exploration.acard_cc_data")

# Drop existing demographic columns to avoid duplicates after join
drop_cols = ["DATE_OF_BIRTH", "AGE", "NATIONALITY", "SEX"]
existing_drop = [c for c in drop_cols if c in df_base.columns]
df_base = df_base.drop(*existing_drop)

df_base.createOrReplaceTempView("_acard_cc_base")

df_final = spark.sql("""
    SELECT
        a.*,
        c.NATIONALITY,
        c.DATE_OF_BIRTH,
        c.SEX,
        FLOOR(MONTHS_BETWEEN(a.APPROVEDATE, c.DATE_OF_BIRTH) / 12) AS AGE
    FROM _acard_cc_base a
    LEFT JOIN (
        SELECT DISTINCT Customer_number, NATIONALITY, DATE_OF_BIRTH, SEX
        FROM mis_automation.silver_fin_db.customer_information
        WHERE Customer_number IS NOT NULL
    ) c
    ON a.CIF = c.Customer_number
""")

# Save to table
df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("mis_automation.ds_exploration.acard_cc_data")
print("Saved to mis_automation.ds_exploration.acard_cc_data")



In [0]:
# spark.sql("SELECT * FROM mis_automation.ds_exploration.acard_cc_data LIMIT 5").display() #mis_automation.datascience_pbg.acard_master
spark.sql("SELECT aecb_income_age_months FROM mis_automation.datascience_pbg.acard_master LIMIT 5").display()

In [0]:
%sql

CREATE TABLE mis_automation.ds_exploration.acard_cc_data_snap_20261104 as SELECT * FROM mis_automation.ds_exploration.acard_cc_data

In [0]:
%sql
SELECT
  DATE_TRUNC('QUARTER', APPROVEDATE) AS approve_quarter,
  COUNT(DISTINCT CREDITCARDNO) AS card_count
FROM mis_automation.ds_exploration.acard_cc_data
GROUP BY DATE_TRUNC('QUARTER', APPROVEDATE)
ORDER BY approve_quarter

In [0]:
# ── End-to-end: build df_prev_names from source tables and save to acard_master_ri for reject inference  ──
# Steps (same logic as cells 29 and 36, run natively in Spark):
#   1. reject_ref INNER JOIN aecb_feature_report_level on aecb_reference_number
#   2. Keep aecb_file_type_policy = 'Thin File'
#   3. (optional) Reject inference filters from cell 30: final_decision = 'Decline'
#      AND dectec_all_reason_descriptions contains 'INTERNAL SCORE DECLINE'
#   4. Drop aecb_emirates_id, aecb_cif (clash with emirates_id / cif from reject_ref)
#   5. Rename AECB columns using the mapping file (col C -> col B) + prefix rule
#   6. Save as Delta table
import pandas as pd
from pyspark.sql import functions as F

SRC_REJECT_REF   = "mis_automation.sa_mart.card_onboarding_reject_reference_dataset"
SRC_AECB         = "mis_automation.sa_mart.aecb_feature_report_level"
MAPPING_PATH     = "/Volumes/mis_automation/datascience_pbg/pbg_cc_ascore/input/AECB_Feature_Column_Mapping.xlsx"
# TARGET_TABLE     = "mis_automation.datascience_pbg.acard_master_ri_fts"
APPLY_RI_FILTERS = False   # False = same rows as df_prev_names (Thin File universe); True = reject inference universe only
RI_RISK_SEGMENTS  = ["Elevated Risk", "Significant Risk"]  # dectec_output_risk_segmentation values for RI universe
DROP_COLS        = ["aecb_emirates_id", "aecb_cif"]

# ── 1-2. Join and Thin File filter ────────────────────────────────────────────
reject_ref = spark.table(SRC_REJECT_REF)
aecb_feat  = spark.table(SRC_AECB)
sdf = (
    reject_ref
    .join(aecb_feat, on="aecb_reference_number", how="inner")
    .filter(F.col("aecb_file_type_policy") == "Thin File")
)

# ── 3. Optional reject inference filters ──────────────────────────────────────
if APPLY_RI_FILTERS:
    sdf = sdf.filter(
        F.col("dectec_output_risk_segmentation").isin(RI_RISK_SEGMENTS)
    )

# ── 4. Drop clashing columns ──────────────────────────────────────────────────
cols = sdf.columns   # read schema once
sdf = sdf.drop(*[c for c in DROP_COLS if c in cols])
cols = [c for c in cols if c not in DROP_COLS]

# ── 5. Build rename map from the mapping file ────────────────────────────────
m = pd.read_excel(MAPPING_PATH, sheet_name="Column_Mapping", header=3).dropna(subset=["New_Column_Name"])
m["New_Column_Name"] = m["New_Column_Name"].astype(str).str.strip()
m["Previous_Column_Name"] = m["Previous_Column_Name"].astype(str).str.strip()

def target_name(new: str, prev: str) -> str:
    """Col B name; '(new column)' keeps col C. Then: AECB_Score -> aecb_AECB_Score,
    names starting with 'aecb_' lose the prefix, all others get the 'aecb_' prefix.
    Special cases: file_type, file_type_new, file_type_revised are returned as-is."""
    t = new if prev.lower() in ("(new column)", "nan", "") else prev
    if t in ("file_type", "file_type_new", "file_type_revised"):
        return t
    if t == "AECB_Score":
        return "aecb_AECB_Score"
    if t.startswith("aecb_"):
        return t[len("aecb_"):]
    return "aecb_" + t

rename_map = {
    new: target_name(new, prev)
    for new, prev in zip(m["New_Column_Name"], m["Previous_Column_Name"])
    if new in cols and target_name(new, prev) != new
}

# Guard: final names must be unique (case-insensitive, as Delta requires)
final_cols = [rename_map.get(c, c) for c in cols]
lower = pd.Series([c.lower() for c in final_cols])
dups = sorted(set(lower[lower.duplicated()]))
if dups:
    raise ValueError(f"Duplicate column names after rename (case-insensitive): {dups}")

sdf = sdf.withColumnsRenamed(rename_map)
print(f"Columns renamed: {len(rename_map)} | dropped: {DROP_COLS} | final columns: {len(final_cols)}")

# # ── 6. Save ───────────────────────────────────────────────────────────────────
# (sdf.write
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(TARGET_TABLE))

# saved = spark.table(TARGET_TABLE)
# print(f"Saved {TARGET_TABLE}: {saved.count():,} rows x {len(saved.columns)} columns (APPLY_RI_FILTERS={APPLY_RI_FILTERS})")
# display(saved.limit(5))

In [0]:
def load_fts_features_ri(sdf_input, journey="digital"):
    """
    Load FTS features and merge them onto the reject-inference Spark DataFrame.

    Parameters
    ----------
    sdf_input : pyspark.sql.DataFrame
        Reject-inference Spark DF with 'fts_ref_no' column to join on.
    journey : str
        Which journey to load. Currently supports 'digital'.

    Returns
    -------
    pyspark.sql.DataFrame – sdf_input enriched with FTS feature columns.
    """
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    # ── 1. Select source table based on journey ──────────────────────────
    if journey == "digital":
        table = "mis_automation.silver_ftsconnect.fts_features"
    else:
        raise ValueError(f"Unknown journey '{journey}'. Supported: 'digital'.")

    fts_features_sdf = spark.sql(f"SELECT * FROM {table}")
    fts_cols_all = fts_features_sdf.columns

    print(f"Journey            : {journey}")
    print(f"sdf_input rows     : {sdf_input.count():,}")
    print(f"fts_features rows  : {fts_features_sdf.count():,}")
    print(f"fts_features cols  : {len(fts_cols_all)}")

    # ── 2. Normalise join key (case-insensitive) ─────────────────────────
    def _norm_key(c):
        k = F.regexp_replace(F.upper(F.trim(F.col(c).cast("string"))), r"\.0$", "")
        return F.when(k.isin("", "NONE", "NAN", "NULL"), F.lit(None)).otherwise(k)

    # Left side: normalise fts_ref_no and emirates_id (EMID)
    left_sdf = (
        sdf_input
        .withColumn("_fts_key", _norm_key("fts_ref_no"))
        .withColumn("_emid_key", _norm_key("emirates_id"))
    )

    # Right side: normalise fts_ref_no and fts_master_id_number
    right_sdf = (
        fts_features_sdf
        .withColumn("_fts_key_r", _norm_key("fts_ref_no"))
        .withColumn("_fts_emid_r", _norm_key("fts_master_id_number"))
        .drop("fts_ref_no")  # avoid case-insensitive clash
        .filter(F.col("_fts_key_r").isNotNull())
    )

    # ── 3. Left-join on the normalised key ───────────────────────────────
    joined_sdf = left_sdf.join(
        right_sdf, left_sdf["_fts_key"] == right_sdf["_fts_key_r"], how="left"
    )

    # ── 4. Resolve duplicate fts_ref_no per input row ────────────────────
    # Prefer the row whose fts_master_id_number matches the applicant EMID,
    # then the latest statement.
    w = Window.partitionBy(
        F.monotonically_increasing_id()  # placeholder; use a real key below
    )
    # Use aecb_reference_number + _fts_key as a partition proxy
    part_cols = ["aecb_reference_number", "_fts_key"]
    w_dedup = Window.partitionBy(*part_cols).orderBy(
        (F.col("_fts_emid_r") == F.col("_emid_key")).cast("int").desc_nulls_last(),
        F.col("fts_statement_create_date").desc_nulls_last(),
    )
    joined_sdf = (
        joined_sdf
        .withColumn("_n_cand", F.count("_fts_key_r").over(Window.partitionBy(*part_cols)))
        .withColumn("_rn", F.row_number().over(w_dedup))
        .filter(F.col("_rn") == 1)
    )

    # ── 5. Report and clean up helper columns ────────────────────────────
    total       = joined_sdf.count()
    n_has_key   = joined_sdf.filter(F.col("_fts_key").isNotNull()).count()
    n_matched   = joined_sdf.filter(F.col("_fts_key_r").isNotNull()).count()
    n_multi     = joined_sdf.filter(F.col("_n_cand") > 1).count()
    n_emid_mis  = joined_sdf.filter(
        F.col("_fts_key_r").isNotNull()
        & F.col("_emid_key").isNotNull()
        & (F.col("_fts_emid_r") != F.col("_emid_key"))
    ).count()

    result = joined_sdf.drop("_fts_key", "_emid_key", "_fts_key_r", "_fts_emid_r", "_n_cand", "_rn")

    existing_cols = set(sdf_input.columns)
    new_cols = [c for c in result.columns if c not in existing_cols]

    print(f"\nResult rows             : {total:,}")
    print(f"Rows with valid FTS key : {n_has_key:,}")
    print(f"Rows matched to FTS     : {n_matched:,} ({n_matched / max(total, 1):.2%} of all, "
          f"{n_matched / max(n_has_key, 1):.2%} of keyed)")
    print(f"Rows with >1 FTS row    : {n_multi:,} (resolved: EMID match, then latest statement)")
    print(f"Matched but EMID differs: {n_emid_mis:,}")
    if new_cols:
        print(f"\nNew columns from fts_features ({len(new_cols)}):")
        for i, c in enumerate(new_cols, 1):
            print(f"  {i:3d}. {c}")
    else:
        print("No new columns added.")

    return result

In [0]:
sdf_ =  load_fts_features_ri(sdf, journey="digital")

In [0]:
def load_fts_features_physical_ri(sdf_input):
    """
    EMID fallback for reject-inference Spark DataFrame: for rows without
    fts_ref_no, match FTS features via EMID = fts_master_id_number.

    Keeps the latest fts_statement_create_date per row within a
    [-60d, +30d) window around dectec_application_date.

    Parameters
    ----------
    sdf_input : pyspark.sql.DataFrame
        Reject-inference Spark DF (already enriched by the digital journey).
        Must contain 'fts_ref_no', 'emirates_id', and 'dectec_application_date'.

    Returns
    -------
    pyspark.sql.DataFrame – sdf_input with FTS columns back-filled for rows
                            that had no fts_ref_no but matched on EMID.
    """
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    # ── 0. Normalise key helper ──────────────────────────────────────────
    def _norm_key(c):
        k = F.regexp_replace(F.upper(F.trim(F.col(c).cast("string"))), r"\.0$", "")
        return F.when(k.isin("", "NONE", "NAN", "NULL"), F.lit(None)).otherwise(k)

    # ── 1. Identify rows where fts_ref_no is missing ─────────────────────
    no_fts_cond = (
        F.col("fts_ref_no").isNull()
        | F.upper(F.trim(F.col("fts_ref_no").cast("string"))).isin("", "NONE", "NAN", "NULL")
    )
    has_fts_cond = ~no_fts_cond

    # Rows that already have FTS data – keep untouched
    sdf_has_fts = sdf_input.filter(has_fts_cond)

    # Rows without FTS_Ref_No but with a valid EMID
    sdf_no_fts = sdf_input.filter(no_fts_cond)

    total_no_fts = sdf_no_fts.count()
    has_emid = sdf_no_fts.filter(F.col("emirates_id").isNotNull()).count()
    print(f"Cases without fts_ref_no: {total_no_fts:,}")
    print(f"  ... with valid emirates_id (EMID): {has_emid:,}")

    if has_emid == 0:
        print("Nothing to backfill — returning sdf unchanged.")
        return sdf_input

    # ── 2. Load FTS features ─────────────────────────────────────────────
    fts_sdf = (
        spark.table("mis_automation.silver_ftsconnect.fts_features")
        .withColumn("_fts_emid_r", _norm_key("fts_master_id_number"))
    )

    # ── 3. FTS feature columns (exclude join keys) ───────────────────────
    fts_feature_cols = [
        c for c in fts_sdf.columns
        if c.startswith("fts_") and c not in ("fts_master_id_number", "fts_ref_no")
    ]

    # ── 4. Prepare no-fts side: normalise EMID key ───────────────────────
    sdf_no_fts = sdf_no_fts.withColumn("_emid_key", _norm_key("emirates_id"))

    # Drop existing FTS feature columns from no-fts side so the join
    # can bring in fresh values without ambiguity
    existing_fts_cols = [c for c in fts_feature_cols if c in sdf_no_fts.columns]
    if existing_fts_cols:
        sdf_no_fts = sdf_no_fts.drop(*existing_fts_cols)

    # Also drop fts_ref_no from fts_sdf to avoid clash (case-insensitive)
    fts_sdf_join = fts_sdf.drop("fts_ref_no", "fts_master_id_number")

    # ── 5. Join on EMID (case-insensitive) ───────────────────────────────
    joined = sdf_no_fts.join(
        fts_sdf_join,
        sdf_no_fts["_emid_key"] == fts_sdf_join["_fts_emid_r"],
        how="left",
    )

    cards_before = joined.filter(F.col("_fts_emid_r").isNotNull()).select("aecb_reference_number").distinct().count()
    print(f"\n── Drop-off Diagnosis ──")
    print(f"Step 5 - Rows matched on EMID (before date filter): {cards_before:,}")

    # ── 6. Date filter: [-60d, +30d) around dectec_application_date ──────
    joined = joined.withColumn(
        "_in_window",
        (
            F.col("fts_statement_create_date").isNotNull()
            & (F.col("fts_statement_create_date") >= F.expr("dectec_application_date - INTERVAL 60 DAYS"))
            & (F.col("fts_statement_create_date") < F.expr("dectec_application_date + INTERVAL 30 DAYS"))
        ),
    )

    # Null-out FTS feature cols for rows outside the window
    for c in fts_feature_cols:
        if c in joined.columns:
            joined = joined.withColumn(
                c, F.when(F.col("_in_window"), F.col(c)).otherwise(F.lit(None))
            )

    cards_after = joined.filter(F.col("_in_window")).select("aecb_reference_number").distinct().count()
    print(f"Step 6 - Rows surviving [-60d, +30d) date filter:   {cards_after:,}")
    print(f"         Rows LOST by date filter:                  {cards_before - cards_after:,}")

    # ── 7. De-duplicate: keep latest fts_statement_create_date per row ───
    w = Window.partitionBy("aecb_reference_number").orderBy(
        F.col("fts_statement_create_date").desc_nulls_last()
    )
    joined = (
        joined
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn", "_emid_key", "_fts_emid_r", "_in_window")
    )

    matched = joined.filter(F.col("fts_statement_create_date").isNotNull()).count()
    print(f"\nMatched via EMID (after dedup): {matched:,} rows")

    # ── 8. Union back with the rows that already had FTS ─────────────────
    # Align columns: ensure both sides have the same schema
    all_cols = list(dict.fromkeys(sdf_has_fts.columns + [c for c in joined.columns if c not in sdf_has_fts.columns]))
    for c in all_cols:
        if c not in sdf_has_fts.columns:
            sdf_has_fts = sdf_has_fts.withColumn(c, F.lit(None))
        if c not in joined.columns:
            joined = joined.withColumn(c, F.lit(None))

    result = sdf_has_fts.select(all_cols).unionByName(joined.select(all_cols))

    # ── Summary ──────────────────────────────────────────────────────────
    total = result.count()
    fts_filled = result.filter(F.col("fts_statement_create_date").isNotNull()).count()
    print(f"\nFinal row count:   {total:,}")
    print(f"FTS coverage:      {fts_filled:,} / {total:,} ({fts_filled / total:.1%})")
    print(f"Rows gained:       {matched:,}")

    return result

In [0]:
sdf__ = load_fts_features_physical_ri(sdf_)

In [0]:
# Diagnose FTS population source in sdf__
from pyspark.sql import functions as F

# Normalise helper (same logic used in the pipeline functions)
def _is_present(col_name):
    k = F.regexp_replace(F.upper(F.trim(F.col(col_name).cast("string"))), r"\.0$", "")
    return ~k.isin("", "NONE", "NAN", "NULL") & F.col(col_name).isNotNull()

total = sdf__.count()
has_fts = sdf__.filter(F.col("fts_statement_create_date").isNotNull()).count()
no_fts  = total - has_fts

# Rows that had a valid fts_ref_no → matched via digital journey
via_ref_no = sdf__.filter(
    _is_present("fts_ref_no") & F.col("fts_statement_create_date").isNotNull()
).count()

# Rows without fts_ref_no but still have FTS data → matched via EMID fallback
via_emid = sdf__.filter(
    ~_is_present("fts_ref_no") & F.col("fts_statement_create_date").isNotNull()
).count()

print(f"Total rows in sdf__          : {total:,}")
print(f"FTS populated (any source)   : {has_fts:,}  ({has_fts/total:.1%})")
print(f"  ├─ via fts_ref_no (digital) : {via_ref_no:,}  ({via_ref_no/total:.1%})")
print(f"  └─ via emirates_id (EMID)   : {via_emid:,}  ({via_emid/total:.1%})")
print(f"No FTS data                  : {no_fts:,}  ({no_fts/total:.1%})")

In [0]:
TARGET_TABLE = "mis_automation.datascience_pbg.acard_master_ri_fts"

# Drop NullType (void) columns — all-null cols that Parquet can't store/read
from pyspark.sql.types import NullType
void_cols = [f.name for f in sdf__.schema.fields if isinstance(f.dataType, NullType)]
if void_cols:
    print(f"Dropping NullType (void) columns: {void_cols}")
    sdf__ = sdf__.drop(*void_cols)

sdf__.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)
print(f"Saved to {TARGET_TABLE}: {spark.table(TARGET_TABLE).count():,} rows x {len(spark.table(TARGET_TABLE).columns)} columns")

display(spark.table(TARGET_TABLE).limit(5))

In [0]:
%sql
ALTER TABLE mis_automation.datascience_pbg.acard_master_ri_fts
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

ALTER TABLE mis_automation.datascience_pbg.acard_master_ri_fts
DROP COLUMN IF EXISTS AGE;

ALTER TABLE mis_automation.datascience_pbg.acard_master_ri_fts
ADD COLUMNS (AGE INT);

UPDATE mis_automation.datascience_pbg.acard_master_ri_fts
SET AGE = FLOOR(DATEDIFF(dectec_application_date, date_of_birth) / 365.25);

SELECT dectec_application_date, date_of_birth, AGE
FROM mis_automation.datascience_pbg.acard_master_ri_fts
LIMIT 10;

In [0]:
# Compare AECB and FTS column names between acard_master_ri_fts and acard_master_ri
ri_fts_cols = set(spark.table("mis_automation.datascience_pbg.acard_master_ri_fts").columns)
ri_cols = set(spark.table("mis_automation.datascience_pbg.acard_master_ri").columns)

# Extract AECB and FTS columns from each table
ri_fts_aecb = sorted(c for c in ri_fts_cols if c.lower().startswith("aecb_"))
ri_aecb     = sorted(c for c in ri_cols     if c.lower().startswith("aecb_"))
ri_fts_fts  = sorted(c for c in ri_fts_cols if c.lower().startswith("fts_"))
ri_fts_only_fts_cols = sorted(c for c in ri_cols if c.lower().startswith("fts_"))

print("=" * 80)
print("AECB COLUMNS COMPARISON")
print("=" * 80)
print(f"acard_master_ri_fts AECB cols: {len(ri_fts_aecb)}")
print(f"acard_master_ri     AECB cols: {len(ri_aecb)}")

aecb_only_in_ri_fts = sorted(set(ri_fts_aecb) - set(ri_aecb))
aecb_only_in_ri     = sorted(set(ri_aecb) - set(ri_fts_aecb))
aecb_common         = sorted(set(ri_fts_aecb) & set(ri_aecb))

print(f"\nCommon AECB cols:              {len(aecb_common)}")
print(f"Only in acard_master_ri_fts:   {len(aecb_only_in_ri_fts)}")
if aecb_only_in_ri_fts:
    for c in aecb_only_in_ri_fts:
        print(f"  + {c}")
print(f"Only in acard_master_ri:       {len(aecb_only_in_ri)}")
if aecb_only_in_ri:
    for c in aecb_only_in_ri:
        print(f"  + {c}")

print("\n" + "=" * 80)
print("FTS COLUMNS COMPARISON")
print("=" * 80)
print(f"acard_master_ri_fts FTS cols:  {len(ri_fts_fts)}")
print(f"acard_master_ri     FTS cols:  {len(ri_fts_only_fts_cols)}")

fts_only_in_ri_fts = sorted(set(ri_fts_fts) - set(ri_fts_only_fts_cols))
fts_only_in_ri     = sorted(set(ri_fts_only_fts_cols) - set(ri_fts_fts))
fts_common         = sorted(set(ri_fts_fts) & set(ri_fts_only_fts_cols))

print(f"\nCommon FTS cols:               {len(fts_common)}")
print(f"Only in acard_master_ri_fts:   {len(fts_only_in_ri_fts)}")
if fts_only_in_ri_fts:
    for c in fts_only_in_ri_fts:
        print(f"  + {c}")
print(f"Only in acard_master_ri:       {len(fts_only_in_ri)}")
if fts_only_in_ri:
    for c in fts_only_in_ri:
        print(f"  + {c}")

print("\n" + "=" * 80)
print("ALL COLUMNS COMPARISON (summary)")
print("=" * 80)
all_only_ri_fts = sorted(ri_fts_cols - ri_cols)
all_only_ri     = sorted(ri_cols - ri_fts_cols)
print(f"Total cols in acard_master_ri_fts: {len(ri_fts_cols)}")
print(f"Total cols in acard_master_ri:     {len(ri_cols)}")
print(f"Common cols:                       {len(ri_fts_cols & ri_cols)}")
print(f"Only in acard_master_ri_fts:       {len(all_only_ri_fts)}")
print(f"Only in acard_master_ri:           {len(all_only_ri)}")